In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime

StatementMeta(, af62311e-3748-4736-96a5-a8e3b5f1e829, 3, Finished, Available, Finished, False)

In [ ]:
# p_file_name          = spark.conf.get("p_file_name")
# p_file_name          = "customers.csv"
# p_default_load_type  = spark.conf.get("p_default_load_type", "full")
# p_config_load_type   = spark.conf.get("load_type", "")


load_type = (
    p_config_load_type.strip().lower()
    if p_config_load_type and p_config_load_type.strip() != ""
    else p_default_load_type.strip().lower()
)

print(f"File Name : {p_file_name}")
print(f"Load Type : {load_type}")

StatementMeta(, af62311e-3748-4736-96a5-a8e3b5f1e829, 4, Finished, Available, Finished, False)

File Name : customers.csv
Load Type : full


# READ RAW FILE 

In [ ]:


df_raw_cust = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f'Files/raw_uploads/{p_file_name}')
)


df_clean_cust = (df_raw_cust
    .filter(F.col("customer_id").isNotNull())           # remove null PKs
    .dropDuplicates(["customer_id"])                    # deduplicate
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("city",          F.trim(F.col("city")))
    .withColumn("segment",       F.trim(F.col("segment")))
)

ingested_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_clean_cust = (
    df_clean_cust
        .withColumn("ingested_at", F.lit(ingested_at))
)

target_table = "clean_customers"
compare_cols = [
    "customer_name",
    "city",
    "segment"
]

StatementMeta(, af62311e-3748-4736-96a5-a8e3b5f1e829, 5, Finished, Available, Finished, False)

In [ ]:
target_table = "silver.clean_customers"
stage_table  = "stage.stg_new_customers"

StatementMeta(, af62311e-3748-4736-96a5-a8e3b5f1e829, 6, Finished, Available, Finished, False)

In [ ]:
df_clean_cust.cache()

StatementMeta(, af62311e-3748-4736-96a5-a8e3b5f1e829, 7, Finished, Available, Finished, False)

DataFrame[customer_id: string, customer_name: string, city: string, segment: string, last_modified: timestamp, ingested_at: string]

In [ ]:
def build_stage_from_batch(df_batch, target_table, compare_cols):

    # First load
    if not spark.catalog.tableExists(target_table):

        return (
            df_batch
                .withColumn("change_type", F.lit("UPSERT"))
        )

    # Get only batch IDs
    batch_ids = df_batch.select("customer_id")

    # Read only matching IDs from target
    df_existing = (
        spark.table(target_table)
            .join(batch_ids, on="customer_id", how="inner")
    )

    # =====================================================
    # NEW ROWS
    # =====================================================

    df_new = (
        df_batch.alias("s")
            .join(
                df_existing.select("customer_id").alias("t"),
                on="customer_id",
                how="left_anti"
            )
            .withColumn("change_type", F.lit("UPSERT"))
    )

    # =====================================================
    # UPDATED ROWS
    # =====================================================

    changed_cond = F.lit(False)

    for col in compare_cols:

        changed_cond = (
            changed_cond |
            (
                F.coalesce(F.col(f"s.{col}"), F.lit("")) !=
                F.coalesce(F.col(f"t.{col}"), F.lit(""))
            )
        )

    df_updated = (
        df_batch.alias("s")
            .join(
                df_existing.alias("t"),
                on="customer_id",
                how="inner"
            )
            .filter(changed_cond)
            .select(
                F.col("s.customer_id"),
                F.col("s.customer_name"),
                F.col("s.city"),
                F.col("s.segment"),
                F.col("s.ingested_at")
            )
            .withColumn("change_type", F.lit("UPDATE"))
    )

    return df_new.unionByName(df_updated)

StatementMeta(, af62311e-3748-4736-96a5-a8e3b5f1e829, 8, Finished, Available, Finished, False)

In [ ]:

# -- FULL LOAD --
if load_type == "full":
    print("Running FULL LOAD...")

    # STAGE TABLE
    
    df_stage = (
        df_clean_cust
            .withColumn("change_type", F.lit("UPSERT"))
    )

    (
        df_stage.write
            .mode("append")
            .format("delta")
            .saveAsTable(stage_table)
    )

    stage_count = df_stage.count()

    print(f"Stage written : {stage_count} rows")

    # CLEAN TABLE
    (
        df_clean_cust.write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(target_table)
    )

    print("FULL LOAD completed")

   
# -------- INCREMENTAL LOAD --------

elif load_type == "incremental":

    print("Running INCREMENTAL LOAD...")

    df_stage = build_stage_from_batch(
        df_clean_cust,
        target_table,
        compare_cols
    )

    stage_count = df_stage.count()

    if stage_count > 0:
        
        print(f"Changes detected : {stage_count} rows")

        (
            df_stage.write
                .mode("append")
                .format("delta")
                .saveAsTable(stage_table)
        )

        print("Stage APPEND completed")

    else:

        print("No changes detected")
    

    if spark.catalog.tableExists(target_table):

        df_target = spark.table(target_table)

        delta_table = DeltaTable.forName(spark, target_table)

        (
            delta_table.alias("t")
            .merge(
                df_clean_cust.alias("s"),
                "t.customer_id = s.customer_id"
            )
            .whenMatchedUpdate(set={
                "customer_name": "s.customer_name",
                "city": "s.city",
                "segment": "s.segment",
                "ingested_at": "s.ingested_at"
            })
            .whenNotMatchedInsert(values={
                "customer_id": "s.customer_id",
                "customer_name": "s.customer_name",
                "city": "s.city",
                "segment": "s.segment",
                "ingested_at": "s.ingested_at"
            })
            .execute()
        )

        print("INCREMENTAL MERGE completed")
    else:
        (
            df_clean_cust.write
                .mode("overwrite")
                .format("delta")
                .saveAsTable(target_table)
        )

        df_stage = (
            df_clean_cust
                .withColumn("change_type", F.lit("UPSERT"))
        )

        (
            df_stage.write
                .mode("append")
                .format("delta")
                .saveAsTable(stage_table)
        )

else:
    raise Exception(f"Unsupported load type : {load_type}")


# CLEAN CACHE
df_clean_cust.unpersist()


StatementMeta(, af62311e-3748-4736-96a5-a8e3b5f1e829, 9, Finished, Available, Finished, False)

Running FULL LOAD...


Stage written : 47 rows


FULL LOAD completed


DataFrame[customer_id: string, customer_name: string, city: string, segment: string, last_modified: timestamp, ingested_at: string]